# Session 3: MCP 서버 연동을 통한 Agent 기능 확장

### Index
1. MCP(Model Context Protocol) 개념
2. MCP 서버 준비 (SQLite)
3. Strands Agent + MCP 연동
4. 자연어로 데이터 질의
5. AgentCore Gateway 연동

---
## 1. MCP(Model Context Protocol)란?

MCP는 AI Agent가 **외부 도구/서비스와 통신하기 위한 표준 프로토콜**입니다.  
USB-C처럼, 어떤 Agent든 MCP 서버에 "연결하면" 바로 기능을 쓸 수 있습니다.

### Session 2와의 차이

| | 직접 Tool (Session 2) | MCP Tool (Session 3) |
|---|---|---|
| Tool 정의 위치 | Agent 코드 안 (Python 함수) | 독립된 MCP 서버 |
| Tool 탐색 방식 | 코드에 미리 등록 | 런타임에 서버에서 동적으로 조회 |
| 재사용 범위 | 같은 Python 프로젝트 | 언어/플랫폼 무관하게 연결 가능 |
| 배포 단위 | Agent와 함께 배포 | 서버 독립 배포/교체 가능 |

### MCP 구조

|  | MCP Client (Agent) | | MCP Server |
|:---:|---|:---:|---|
| ① | Tool 목록 요청 | ---> | Tool 목록 반환 |
| ② | Tool 목록 수신 | <--- | Tool 목록 전달 |
| ③ | Tool 호출 | ---> | Tool 실행 |
| ④ | 결과 수신 | <--- | 실행 결과 반환 |

- **Transport**: 오늘은 **stdio** (로컬 프로세스 간 통신), 원격 서버는 HTTP+SSE 사용

> 💡 MCP 서버는 이미 다양한 오픈소스가 있어서, 직접 만들지 않아도 바로 연결할 수 있습니다.  
> 오늘은 `mcp-server-sqlite`를 사용합니다.

---
## 2. MCP 서버 준비 (SQLite)

> SQLite MCP 서버를 사용하면 Agent가 **자연어로 DB를 질의**할 수 있습니다.  
> 먼저 샘플 데이터베이스를 만들어보겠습니다.

In [ ]:
# ! pip install strands-agents mcp

### 샘플 데이터베이스 생성

> 간단한 쇼핑몰 데이터를 만듭니다: 상품(products), 주문(orders)

In [ ]:
import sqlite3
import os

# DB 파일 경로
DB_PATH = "data/shop.db"

# 기존 파일 삭제 후 새로 생성
if os.path.exists(DB_PATH):
  os.remove(DB_PATH)

os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# 상품 테이블
cursor.execute("""
CREATE TABLE products (
  id INTEGER PRIMARY KEY,
  name TEXT NOT NULL,
  category TEXT NOT NULL,
  price INTEGER NOT NULL,
  stock INTEGER NOT NULL
)
""")

# 주문 테이블
cursor.execute("""
CREATE TABLE orders (
  id INTEGER PRIMARY KEY,
  product_id INTEGER,
  quantity INTEGER NOT NULL,
  order_date TEXT NOT NULL,
  customer_name TEXT NOT NULL,
  FOREIGN KEY (product_id) REFERENCES products(id)
)
""")

# 샘플 상품 데이터
products = [
  (1, "맥북 프로 14", "전자기기", 2800000, 15),
  (2, "에어팟 프로", "전자기기", 359000, 50),
  (3, "파이썬 코딩의 기술", "도서", 32000, 100),
  (4, "AWS 교과서", "도서", 38000, 80),
  (5, "기계식 키보드", "전자기기", 150000, 30),
  (6, "모니터 암", "사무용품", 45000, 60),
  (7, "스탠딩 데스크", "사무용품", 580000, 10),
  (8, "노이즈캔슬링 헤드폰", "전자기기", 420000, 25),
]
cursor.executemany("INSERT INTO products VALUES (?, ?, ?, ?, ?)", products)

# 샘플 주문 데이터
orders = [
  (1, 1, 1, "2026-05-01", "김철수"),
  (2, 2, 2, "2026-05-01", "이영희"),
  (3, 3, 3, "2026-05-02", "박민수"),
  (4, 4, 1, "2026-05-02", "김철수"),
  (5, 1, 1, "2026-05-03", "정다은"),
  (6, 5, 2, "2026-05-03", "이영희"),
  (7, 2, 1, "2026-05-04", "최준호"),
  (8, 7, 1, "2026-05-04", "박민수"),
  (9, 6, 3, "2026-05-05", "김철수"),
  (10, 8, 1, "2026-05-05", "정다은"),
]
cursor.executemany("INSERT INTO orders VALUES (?, ?, ?, ?, ?)", orders)

conn.commit()
conn.close()

print(f"✅ 데이터베이스 생성 완료: {DB_PATH}")
print(f"   - products: {len(products)}개 상품")
print(f"   - orders: {len(orders)}개 주문")

### 데이터 확인

In [ ]:
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

print("=== 상품 목록 ===")
for row in cursor.execute("SELECT * FROM products"):
  print(row)

print("\n=== 최근 주문 5건 ===")
for row in cursor.execute("SELECT o.id, p.name, o.quantity, o.order_date, o.customer_name FROM orders o JOIN products p ON o.product_id = p.id ORDER BY o.order_date DESC LIMIT 5"):
  print(row)

conn.close()

---
## 3. Strands Agent + MCP 연동

> `mcp-server-sqlite`를 MCP Client로 연결하고, Agent에 붙입니다.  
> Agent는 MCP 서버가 제공하는 Tool 목록을 자동으로 인식합니다.

### mcp-server-sqlite가 제공하는 Tool들

| 분류 | Tool | 설명 |
|------|------|------|
| Query | `read_query` | SELECT 쿼리 실행 (읽기 전용) |
| Query | `write_query` | INSERT/UPDATE/DELETE 실행 |
| Query | `create_table` | 테이블 생성 (CREATE TABLE) |
| Schema | `list_tables` | 데이터베이스의 모든 테이블 목록 조회 |
| Schema | `describe_table` | 특정 테이블의 컬럼 정보 조회 |
| Analysis | `append_insight` | 분석 중 발견한 비즈니스 인사이트를 메모에 추가 |

In [ ]:
from mcp import stdio_client, StdioServerParameters
from strands import Agent
from strands.tools.mcp import MCPClient
from strands.models import BedrockModel

In [ ]:
model = BedrockModel(
  model_id="us.amazon.nova-2-lite-v1:0",
  region_name="us-east-1"
)

In [ ]:
# MCP Client 생성 — SQLite MCP 서버에 연결
sqlite_mcp = MCPClient(lambda: stdio_client(
  StdioServerParameters(
      command="uvx",
      args=["mcp-server-sqlite", "--db-path", "./data/shop.db"]
  )
))

In [ ]:
# MCP Client를 context manager로 사용하여 Agent 생성

agent = Agent(
    model=model,
    tools=[sqlite_mcp],
    system_prompt="""너는 쇼핑몰 데이터 분석가야. SQLite 데이터베이스에 연결되어 있어. 사용자의 질문에 SQL 쿼리를 실행해서 정확한 데이터를 기반으로 답변해. 한국어로 답변해."""
)

# MCP 서버가 제공하는 Tool 목록 확인
print("📋 사용 가능한 Tool 목록:")
print("=" * 50)
for tool_name, tool in agent.tool_registry.registry.items():
    desc = tool.tool_spec.get('description', '') if hasattr(tool, 'tool_spec') else ''
    print(f"  - {tool_name}: {desc[:60]}")

---
## 4. 자연어로 데이터 질의

> 이제 Agent에게 자연어로 질문하면, Agent가 알아서:  
> 1. 테이블 구조를 파악하고  
> 2. 적절한 SQL을 생성하고  
> 3. 실행 결과를 해석해서 답변합니다.

In [ ]:
system_prompt="""너는 쇼핑몰 데이터 분석가야. SQLite 데이터베이스에 연결되어 있어. 사용자의 질문에 SQL 쿼리를 실행해서 정확한 데이터를 기반으로 답변해. 한국어로 답변해."""

In [ ]:
# 기본 조회
agent = Agent(model=model, tools=[sqlite_mcp], system_prompt=system_prompt)

response = agent("어떤 테이블이 있고, 각 테이블에 어떤 컬럼이 있는지 알려줘")

In [ ]:
# 집계 질의
agent = Agent(model=model, tools=[sqlite_mcp], system_prompt=system_prompt)

response = agent("카테고리별 상품 수와 평균 가격을 알려줘")

In [ ]:
# JOIN 질의
agent = Agent(model=model, tools=[sqlite_mcp], system_prompt=system_prompt)

response = agent("가장 많이 주문한 고객은 누구야? 총 주문 금액도 알려줘")

In [ ]:
# 분석 질의
agent = Agent(model=model, tools=[sqlite_mcp], system_prompt=system_prompt)

response = agent("5월 3일 이후 주문 중에서 전자기기 카테고리 매출 합계는?")

In [ ]:
# 인사이트 요청
agent = Agent(model=model, tools=[sqlite_mcp], system_prompt=system_prompt)

response = agent("재고가 20개 이하인 상품 중에서 주문이 많은 상품은? 재입고 우선순위를 추천해줘")

# AgentCore Gateway 연동

In [ ]:
from mcp import stdio_client, StdioServerParameters
from strands import Agent
from strands.tools.mcp import MCPClient
from strands.models import BedrockModel
from mcp_proxy_for_aws.client import aws_iam_streamablehttp_client

In [ ]:
GATEWAY_URL = "https://<gateway-id>.gateway.bedrock-agentcore.<region>.amazonaws.com/mcp"
AWS_REGION = "us-east-1"

In [ ]:
def get_gateway_mcp_client() -> MCPClient:
  """SigV4(IAM) 인증으로 AgentCore Gateway에 연결하는 MCP 클라이언트"""
  return MCPClient(lambda: aws_iam_streamablehttp_client(
      endpoint=GATEWAY_URL,
      aws_region=AWS_REGION,
      aws_service="bedrock-agentcore",
  ))

gateway_mcp = get_gateway_mcp_client()

In [ ]:
# MCP Client 생성 — SQLite MCP 서버에 연결
sqlite_mcp = MCPClient(lambda: stdio_client(
  StdioServerParameters(
      command="uvx",
      args=["mcp-server-sqlite", "--db-path", "./data/shop.db"]
  )
))

In [ ]:
# MCP Client 생성 — SQLite MCP 서버에 연결
with gateway_mcp, sqlite_mcp:
  gateway_tools = gateway_mcp.list_tools_sync()
  sqlite_tools = sqlite_mcp.list_tools_sync()
  all_tools = gateway_tools + sqlite_tools

  print("📋 사용 가능한 Tool 목록:")
  print("=" * 50)
  print("[Gateway]")
  for tool in gateway_tools:
      desc = tool.tool_spec.get('description', '') if hasattr(tool, 'tool_spec') else ''
      print(f"  - {tool.tool_name}: {desc[:60]}")

  print("[SQLite]")
  for tool in sqlite_tools:
      desc = tool.tool_spec.get('description', '') if hasattr(tool, 'tool_spec') else ''
      print(f"  - {tool.tool_name}: {desc[:60]}")

In [ ]:
model = BedrockModel(
  model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0",
  region_name="us-east-1"
)

In [ ]:
with gateway_mcp, sqlite_mcp:
    agent = Agent(
        model=model,
        tools=all_tools,
        system_prompt="""너는 쇼핑몰 데이터 분석가야. SQLite 데이터베이스와 Gateway에 연결된 도구를 모두 사용할 수 있어. 사용자의 질문에 적절한 도구를 선택해서 정확한 데이터를 기반으로 답변해. 한국어로 답변해."""
    )
    
    response = agent("매출 데이터를 조회하고, 메일로 전송해줘.")
    print(response.stop_reason)